# 01 · Participant quality control (before any classification)

For every store: label-blind participant summaries, **category A** technical corruption (blocks the experiments
until a decision is recorded), **category B** statistically unusual but valid EEG (robust modified z-scores against
the 52 development participants, multivariate MCD outliers; flagged and kept). Category C (difficult to classify)
follows after the experiments (notebook 99). **No participant is excluded by any rule**: a decision other than
`keep` stops the experiments, because an exclusion changes the frozen split and must be a logged protocol amendment.
Decisions go in `FINAL_v1/qc/decisions.yaml` (format in `docs/final/qc.md`). CPU runtime.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move the release code pin deliberately to the newest commit
                      # (<outputs>/FINAL_v1/CODE_COMMIT.txt; unfinished work is then recomputed).
BRANCH = 'claude/unified-final-framework'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)

## 1. QC of the four stores (computed once, then loaded)

In [ ]:
UNRESOLVED = []
for inverse in ('lcmv', 'wmne'):
    for mode in ('B', 'A'):
        try:
            sh(f"python scripts/final.py qc --inverse {inverse} --mode {mode}")
        except RuntimeError:
            UNRESOLVED.append((inverse, mode))

## 2. Reports

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None
from IPython.display import Markdown
for d in sorted(glob.glob(FINAL + '/qc/*_*_*/')):
    if os.path.exists(d + 'qc_report.md'):
        display(Markdown(open(d + 'qc_report.md').read()))
        if os.path.exists(d + 'pca.png'):
            display(Image(filename=d + 'pca.png'))
if UNRESOLVED:
    raise RuntimeError(f'category A findings need a decision for {UNRESOLVED}: see the reports above and '
                       'docs/final/qc.md; experiments refuse to start until then')